# Part C — Recurrent Classifiers on AG News & IMDB (Colab / CUDA T4)

> **Disclaimer.** This notebook **re-runs** every Part C experiment from
> scratch on a CUDA T4 (Colab GPU runtime). The local notebook
> `notebooks/part_c_rnn.ipynb` only loads and visualises the pre-computed
> JSONs; this one regenerates them.

It mirrors the structure of the local notebook section-for-section:

| § | Experiment | Notes |
|---|---|---|
| C.1 | Baseline grid (max_words=25, AG News) | grid over 6 model configs × 3 seeds |
| C.2 | CPU vs GPU speed | 1RNN + 1LSTM, 5 epochs, single seed |
| C.3 | t-SNE of learned embeddings | 1 × 1RNN seed=0, 15 epochs |
| C.4 | Longer sequences (max_words=50) | grid |
| C.5 | Pre-trained word2vec (frozen=False) | grid |
| C.6 | Pre-trained word2vec (frozen=True)  | grid |
| C.7 | IMDB instead of AG News | grid |

Every grid cell calls `run_grid(...)` from `experiments.py` which is
**checkpointed**: the JSON on disk is updated after every `(config × seed)`
run, so a Colab disconnect can be resumed simply by re-executing the cell.


## 1. Setup

### 1.1 GPU sanity check

In [1]:
import torch

assert torch.cuda.is_available(), (
    "Enable Runtime > Change runtime type > GPU (T4) before running this notebook."
)
print("GPU    :", torch.cuda.get_device_name(0))
print("CUDA   :", torch.version.cuda)
print("PyTorch:", torch.__version__)


GPU    : Tesla T4
CUDA   : 12.8
PyTorch: 2.10.0+cu128


### 1.2 Clone the repository

Shallow clone keeps the download to a few MB.

In [2]:
!git clone --depth 1 https://github.com/Fgram-devAI/nlp-ncsr-task2.git
%cd nlp-ncsr-task2


Cloning into 'nlp-ncsr-task2'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 48 (delta 12), reused 44 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 669.86 KiB | 5.72 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/nlp-ncsr-task2


### 1.3 Install Part C requirements

`torch` and `numpy` are already present in the Colab base image; we only need
the extras that the Part C modules import.


In [4]:
# Colab already ships pandas/numpy/scikit-learn/matplotlib/tqdm/torch.
# Avoid --upgrade — it pulls pandas 3.0 which conflicts with google-colab and other pre-installed packages.
!pip install -q kagglehub gensim


### 1.4 Kaggle credentials

Required because `data.load_imdb()` (used in C.7) downloads the IMDB dataset
through `kagglehub`. Generate `kaggle.json` once at
**kaggle.com → Account → Create new API token**, then upload it here.


In [8]:
!pip install -q datasets

import sys, os
PART_C = os.path.abspath('part_c_rnn_classification')
if PART_C not in sys.path:
    sys.path.insert(0, PART_C)

import data as data_mod
from data import TextDataset
from datasets import load_dataset, concatenate_datasets


def _hf_load_ag_news():
    ds = load_dataset('ag_news')          # already 0-indexed labels
    return TextDataset(
        X_train=list(ds['train']['text']),
        y_train=list(ds['train']['label']),
        X_test =list(ds['test']['text']),
        y_test =list(ds['test']['label']),
        label_names=['World', 'Sports', 'Business', 'Sci/Tech'],
    )


def _hf_load_imdb():
    # HF imdb ships a 25k/25k split; the assignment asks for 80/20 of 50k.
    # Combine and re-split deterministically (seed=42, same as the local loader).
    ds = load_dataset('imdb')
    full = concatenate_datasets([ds['train'], ds['test']]).shuffle(seed=42)
    split = full.train_test_split(test_size=0.20, seed=42)
    return TextDataset(
        X_train=list(split['train']['text']),
        y_train=list(split['train']['label']),
        X_test =list(split['test']['text']),
        y_test =list(split['test']['label']),
        label_names=['negative', 'positive'],
    )


data_mod.load_ag_news = _hf_load_ag_news
data_mod.load_imdb    = _hf_load_imdb
print("HF loaders patched in. AG News + IMDB will download from HuggingFace Hub.")


HF loaders patched in. AG News + IMDB will download from HuggingFace Hub.


### 1.5 Wire `part_c_rnn_classification` into `sys.path` and import the helpers

We import from the repo's existing modules — **nothing in this notebook
re-implements data loading, model construction, or training**.


In [12]:
import sys, os
from pathlib import Path

# Use PROJECT ROOT, not part_c_rnn_classification — relative imports inside
# experiments.py expect to be loaded as part of the package.
ROOT = os.path.abspath('.')          # we are in /content/nlp-ncsr-task2 after %cd
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from part_c_rnn_classification.experiments import (
    GridParams,
    MODEL_CONFIGS,
    run_grid,
    summarize,
    print_summary_table,
)
from part_c_rnn_classification.train import get_default_device

device = get_default_device()        # cuda > mps > cpu — should pick CUDA here
print("device :", device)

RESULTS = Path('part_c_rnn_classification/results')
RESULTS.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = [c['name'] for c in MODEL_CONFIGS]
SEEDS = [0, 1, 2]
print("models :", MODEL_ORDER)
print("seeds  :", SEEDS)


device : cuda
models : ['1RNN', '1Bi-RNN', '2Bi-RNN', '1LSTM', '1Bi-LSTM', '2Bi-LSTM']
seeds  : [0, 1, 2]


## 2. Experiments

### C.1 — Baseline grid (AG News, `max_words=25`, no pretraining)

Trains all 6 model configurations (`MODEL_CONFIGS`) × 3 seeds = 18 runs.
Hyperparameters are the assignment-fixed ones: `epochs=15`, `batch_size=1024`,
`embedding_dim=100`, `hidden_dim=64`, `lr=1e-3`, `min_freq=10`.


In [ ]:
params = GridParams(
    max_words=25,
    epochs=15,
    batch_size=1024,
    embedding_dim=100,
    hidden_dim=64,
    learning_rate=1e-3,
    min_freq=10,
    pretrained=False,
    frozen_embedding=False,
    dataset='ag_news',
)

results = run_grid(
    MODEL_CONFIGS, SEEDS, params,
    device=device,
    out_path=RESULTS / 'c1_baseline_colab.json',
)
print_summary_table(summarize(results), MODEL_ORDER)


device: cuda
params: GridParams(max_words=25, epochs=15, batch_size=1024, embedding_dim=100, hidden_dim=64, learning_rate=0.001, min_freq=10, pretrained=False, frozen_embedding=False, dataset='ag_news')
Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
dataset: ag_news  train=120000  test=7600
vocab size: 19635 (min_freq=10)

▶ 1RNN  seed=0  device=cuda
  epoch  1/15  train_loss=1.2806
  epoch  2/15  train_loss=0.7920
  epoch  3/15  train_loss=0.5469
  epoch  4/15  train_loss=0.4279
  epoch  5/15  train_loss=0.3682
  epoch  6/15  train_loss=0.3263
  epoch  7/15  train_loss=0.2936
  epoch  8/15  train_loss=0.2663
  epoch  9/15  train_loss=0.2463
  epoch 10/15  train_loss=0.2263
  epoch 11/15  train_loss=0.2109
  epoch 12/15  train_loss=0.1924
  epoch 13/15  train_loss=0.1809
  epoch 14/15  train_loss=0.1655
  epoch 15/15  train_loss=0.1541
  ✓ saved checkpoint: part_c_rnn_classification/results/c1_baseline_colab.json (1 runs)

▶ 1RNN  seed=1  device=cu

**Commentary (C.1).** _Add a few sentences here once the cell completes:
which model wins, how big the seed-to-seed std is, where bidirectional /
2-layer variants land vs 1-layer baselines, etc._


### C.2 — CPU vs GPU

Trains the **1RNN** and **1LSTM** configurations for 5 epochs on **CPU** and
on **CUDA**, single seed (`seed=0`), and compares per-epoch wall-clock.




In [ ]:
import time
import torch

# -- 1) Pick the two configs we benchmark
configs_c2 = [c for c in MODEL_CONFIGS if c['name'] in ('1RNN', '1LSTM')]
assert len(configs_c2) == 2, f"Expected 1RNN and 1LSTM in MODEL_CONFIGS"

params_c2 = GridParams(
    max_words=25, epochs=5, batch_size=1024,
    embedding_dim=100, hidden_dim=64, learning_rate=1e-3,
    min_freq=10, pretrained=False, frozen_embedding=False,
    dataset='ag_news',
)

# -- 2) Run on CPU
t0 = time.time()
cpu_results = run_grid(
    configs_c2, [0], params_c2,
    device=torch.device('cpu'),
    out_path=RESULTS / 'c2_cpu_colab.json',
)
print(f"CPU pass total: {time.time() - t0:.1f}s")

# -- 3) Run on CUDA
t0 = time.time()
cuda_results = run_grid(
    configs_c2, [0], params_c2,
    device=torch.device('cuda'),
    out_path=RESULTS / 'c2_cuda_colab.json',
)
print(f"CUDA pass total: {time.time() - t0:.1f}s")

# -- 4) Comparison table
cpu_secs  = {r['name']: r['sec_per_epoch'] for r in cpu_results}
cuda_secs = {r['name']: r['sec_per_epoch'] for r in cuda_results}

print()
header = f"{'model':<8}{'CPU s/ep':>12}{'CUDA s/ep':>12}{'speed-up':>12}"
print(header); print('-' * len(header))
for name in ('1RNN', '1LSTM'):
    c, g = cpu_secs[name], cuda_secs[name]
    print(f"{name:<8}{c:>12.2f}{g:>12.2f}{c/g:>11.1f}x")


**Commentary (C.2).** _Note the speed-up factors. T4 should crush both CPU
and Apple-Silicon MPS for these tiny RNN models — but also note that the
overhead of per-step CUDA kernel launches dampens the gain at this small
batch / seq length._


### C.3 — t-SNE of learned word embeddings

Trains a single **1RNN** for 15 epochs (seed=0, no pretraining) on AG News,
extracts the learned `embedding.weight` matrix, and projects the same
**28 anchor words used in A.6** to 2-D with t-SNE.

If `results/c3_tsne_colab.png` already exists, the cell just re-displays it.


In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from data import load_ag_news, build_vocab, make_loaders
from model import RecurrentClassifier
from train import train_one_run

# Same 28 anchor words as Part A.6 — 7 from each AG News class
ANCHOR_WORDS = [
    # World / politics
    'government', 'president', 'election', 'country', 'foreign', 'war', 'minister',
    # Sports
    'team', 'game', 'player', 'win', 'championship', 'coach', 'score',
    # Business
    'company', 'market', 'stock', 'price', 'profit', 'business', 'economy',
    # Sci / Tech
    'computer', 'software', 'technology', 'internet', 'system', 'data', 'research',
]
assert len(ANCHOR_WORDS) == 28
WORD_CLASS = (['World'] * 7 + ['Sports'] * 7 + ['Business'] * 7 + ['Sci/Tech'] * 7)

fig_path = RESULTS / 'c3_tsne_colab.png'
if fig_path.exists():
    print(f"{fig_path} already exists — skipping training, re-displaying.")
    from PIL import Image
    plt.figure(figsize=(9, 7))
    plt.imshow(Image.open(fig_path)); plt.axis('off'); plt.show()
else:
    # 1) Pick the 1RNN config
    cfg_1rnn = next(c for c in MODEL_CONFIGS if c['name'] == '1RNN')

    params_c3 = GridParams(
        max_words=25, epochs=15, batch_size=1024,
        embedding_dim=100, hidden_dim=64, learning_rate=1e-3,
        min_freq=10, pretrained=False, frozen_embedding=False,
        dataset='ag_news',
    )

    # 2) Train one run. train_one_run trains a fresh model and returns a
    #    result dict; we ask it to expose the trained model so we can read
    #    its embedding matrix. If your local train_one_run uses a different
    #    keyword (e.g. `return_model=True`), adjust here.
    out = train_one_run(cfg_1rnn, params_c3, seed=0, device=device)
    if isinstance(out, tuple):
        result, trained_model = out
    elif isinstance(out, dict) and 'model' in out:
        result, trained_model = out, out['model']
    else:
        raise RuntimeError(
            "train_one_run did not expose the trained model. "
            "Either patch it to return (result, model) or set return_model=True."
        )

    # 3) Need the same vocab the model was trained with
    train_data, val_data, test_data = load_ag_news()
    vocab = build_vocab(train_data, min_freq=10)

    emb = trained_model.embedding.weight.detach().cpu().numpy()

    # 4) Look up the anchor words; drop any that ended up <unk>
    idxs, kept_words, kept_classes = [], [], []
    for w, cls in zip(ANCHOR_WORDS, WORD_CLASS):
        if w in vocab:
            idxs.append(vocab[w]); kept_words.append(w); kept_classes.append(cls)
        else:
            print(f"  - skipping OOV anchor: {w}")
    sub = emb[idxs]

    # 5) t-SNE — perplexity tuned for ~28 points
    perp = max(2, min(10, len(sub) - 1))
    tsne = TSNE(n_components=2, perplexity=perp, init='pca', random_state=0,
                learning_rate='auto')
    proj = tsne.fit_transform(sub)

    # 6) Plot
    colors = {'World': '#1f77b4', 'Sports': '#2ca02c',
              'Business': '#ff7f0e', 'Sci/Tech': '#d62728'}
    plt.figure(figsize=(9, 7))
    for cls in colors:
        mask = [c == cls for c in kept_classes]
        if any(mask):
            xs = [proj[i, 0] for i, m in enumerate(mask) if m]
            ys = [proj[i, 1] for i, m in enumerate(mask) if m]
            plt.scatter(xs, ys, c=colors[cls], label=cls, s=80, alpha=0.8)
    for i, w in enumerate(kept_words):
        plt.annotate(w, (proj[i, 0], proj[i, 1]),
                     xytext=(4, 4), textcoords='offset points', fontsize=9)
    plt.legend(loc='best')
    plt.title('t-SNE of 1RNN embeddings (15 epochs, seed=0, AG News)')
    plt.tight_layout()
    plt.savefig(fig_path, dpi=140, bbox_inches='tight')
    plt.show()
    print(f"saved -> {fig_path}")


**Commentary (C.3).** _Comment on whether the four AG-News classes form
visible clusters; pre-training would tighten them, randomly-initialised 100-d
embeddings learned only via classification loss usually don't separate them
crisply._


### C.4 — Longer input sequences (`max_words=50`)

Same grid as C.1 but feeds 50 tokens per article instead of 25 — directly
tests whether the recurrent variants benefit from more context.


In [ ]:
params = GridParams(
    max_words=50,
    epochs=15,
    batch_size=1024,
    embedding_dim=100,
    hidden_dim=64,
    learning_rate=1e-3,
    min_freq=10,
    pretrained=False,
    frozen_embedding=False,
    dataset='ag_news',
)

results = run_grid(
    MODEL_CONFIGS, SEEDS, params,
    device=device,
    out_path=RESULTS / 'c4_maxwords50_colab.json',
)
print_summary_table(summarize(results), MODEL_ORDER)


**Commentary (C.4).** _Compare to C.1: which models exploit the extra
context, and which already saturated at 25 tokens?_


### C.5 — Pre-trained word2vec embeddings, **trainable**

Initialises the embedding layer from word2vec (via `gensim`) but lets the
embeddings keep updating during training (`frozen_embedding=False`).


In [ ]:
params = GridParams(
    max_words=25,
    epochs=15,
    batch_size=1024,
    embedding_dim=100,
    hidden_dim=64,
    learning_rate=1e-3,
    min_freq=10,
    pretrained=True,
    frozen_embedding=False,
    dataset='ag_news',
)

results = run_grid(
    MODEL_CONFIGS, SEEDS, params,
    device=device,
    out_path=RESULTS / 'c5_pretrained_trainable_colab.json',
)
print_summary_table(summarize(results), MODEL_ORDER)


**Commentary (C.5).** _Pre-training should give a head start, especially
for models that struggled in C.1; magnitude of the gain is the interesting bit._


### C.6 — Pre-trained word2vec embeddings, **frozen**

Same word2vec init as C.5, but `embedding.weight.requires_grad = False`.
Isolates the contribution of the recurrent layers given a fixed
representation.


In [ ]:
params = GridParams(
    max_words=25,
    epochs=15,
    batch_size=1024,
    embedding_dim=100,
    hidden_dim=64,
    learning_rate=1e-3,
    min_freq=10,
    pretrained=True,
    frozen_embedding=True,
    dataset='ag_news',
)

results = run_grid(
    MODEL_CONFIGS, SEEDS, params,
    device=device,
    out_path=RESULTS / 'c6_pretrained_frozen_colab.json',
)
print_summary_table(summarize(results), MODEL_ORDER)


**Commentary (C.6).** _Trainable parameters drop sharply (no embedding
gradients); accuracy usually drops too, but by less than you'd guess._


### C.7 — IMDB binary sentiment

Identical grid to C.1 but on IMDB (binary, longer reviews). `dataset='imdb'`
makes `data.load_imdb()` (kagglehub) fetch the data — hence the kaggle.json
upload above.


In [ ]:
params = GridParams(
    max_words=25,
    epochs=15,
    batch_size=1024,
    embedding_dim=100,
    hidden_dim=64,
    learning_rate=1e-3,
    min_freq=10,
    pretrained=False,
    frozen_embedding=False,
    dataset='imdb',
)

results = run_grid(
    MODEL_CONFIGS, SEEDS, params,
    device=device,
    out_path=RESULTS / 'c7_imdb_colab.json',
)
print_summary_table(summarize(results), MODEL_ORDER)


**Commentary (C.7).** _IMDB reviews are longer than AG-News headlines, so
truncating at 25 tokens is harsh — note whether models cluster differently
than on AG-News._


## 3. Cross-experiment summary

Loads each grid experiment's JSON (preferring the freshly-computed
`*_colab.json`, falling back to the local pre-computed JSON if missing) and
prints two side-by-side tables:

1. **Mean test accuracy ± std** — rows = 6 models, cols = 5 experiments.
2. **Trainable parameter count** — same shape.


In [ ]:
import json

EXPERIMENTS = [
    ('C.1',  'c1_baseline_colab.json',              'c1_baseline.json'),
    ('C.4',  'c4_maxwords50_colab.json',            'c4_maxwords50.json'),
    ('C.5',  'c5_pretrained_trainable_colab.json',  'c5_pretrained_trainable.json'),
    ('C.6',  'c6_pretrained_frozen_colab.json',     'c6_pretrained_frozen.json'),
    ('C.7',  'c7_imdb_colab.json',                  'c7_imdb.json'),
]

def _load_with_fallback(colab_name, local_name):
    p_colab = RESULTS / colab_name
    p_local = RESULTS / local_name
    if p_colab.exists():
        return json.loads(p_colab.read_text()), str(p_colab)
    if p_local.exists():
        print(f"  (using local fallback for {colab_name})")
        return json.loads(p_local.read_text()), str(p_local)
    raise FileNotFoundError(f"Neither {p_colab} nor {p_local} exists.")

per_exp = {}
for label, colab_name, local_name in EXPERIMENTS:
    rows, src = _load_with_fallback(colab_name, local_name)
    per_exp[label] = summarize(rows)
    print(f"  {label}  <- {src}")

# --- 1) accuracy table ---
def _get(summary_row, *keys):
    for k in keys:
        if k in summary_row:
            return summary_row[k]
    return None

print()
print("Mean test accuracy ± std")
print()
header = f"{'model':<8}" + "".join(f"{lab:>18}" for lab, _, _ in EXPERIMENTS)
print(header)
print('-' * len(header))
for name in MODEL_ORDER:
    cells_str = [f"{name:<8}"]
    for lab, _, _ in EXPERIMENTS:
        s = per_exp[lab].get(name, {})
        mean = _get(s, 'test_acc_mean', 'mean_acc', 'mean_test_acc', 'acc_mean')
        std  = _get(s, 'test_acc_std',  'std_acc',  'std_test_acc',  'acc_std')
        if mean is None:
            cells_str.append(f"{'-':>18}")
        else:
            cells_str.append(f"{100 * mean:>10.2f} ±{100 * (std or 0):>5.2f}")
    print(''.join(cells_str))

# --- 2) trainable-parameters table ---
print()
print("Trainable parameters")
print()
print(header)
print('-' * len(header))
for name in MODEL_ORDER:
    cells_str = [f"{name:<8}"]
    for lab, _, _ in EXPERIMENTS:
        s = per_exp[lab].get(name, {})
        n = _get(s, 'trainable_params', 'num_trainable_params', 'n_params', 'params')
        cells_str.append(f"{n:>18,}" if n is not None else f"{'-':>18}")
    print(''.join(cells_str))


## 4. Save results back

Zips up every `*_colab.json` (and the t-SNE PNG if present) and triggers a
browser download — drop this into the repo's
`part_c_rnn_classification/results/` folder to have the Colab-side numbers
locally.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('colab_results', 'zip', RESULTS)
print("created", archive)
files.download(archive)
